<a href="https://colab.research.google.com/github/OlesiaLelet/A-B-testing/blob/main/a_b_testing_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv('/content/drive/MyDrive/bq-results-20260610-083816-1781080753606/bq-results_10.06.csv')
print(df.head(10))

# Фільтруємо початкову таблицю і отримуємо масив значень event_name, які ми хочемо проаналізувати.
filtered_df = df[df['event_name'].isin(["add_payment_info", "add_shipping_info", "begin_checkout", "new account"])]
metrics_names = filtered_df['event_name'].unique()

# Створюємо цикл, який відпрацьовуввтиме для кожного a/b тесту та записуватиме результат у змінну results.
results = []
test_numbers = sorted(df['test'].unique())

for test_number in test_numbers:
  test_df = df[df['test'] == test_number]
  control = test_df[test_df["test_group"] == 1]
  experiment = test_df[test_df["test_group"] == 2]
  control_sessions = control[control["event_name"] == "session"]["value"].sum()
  experiment_sessions = experiment[experiment["event_name"] == "session"]['value'].sum()
  for metric in metrics_names:
    control_conversions = control[control["event_name"] == metric]["value"].sum()
    experiment_conversions = experiment[experiment["event_name"] == metric]["value"].sum()
    control_cn = control_conversions / control_sessions
    experiment_cn = experiment_conversions / experiment_sessions
    metric_change = ((experiment_cn - control_cn) / control_cn)* 100
    # z-test
    conversions = [control_conversions, experiment_conversions]
    sessions  = [control_sessions, experiment_sessions]
    z_stat, p_value = proportions_ztest(conversions, sessions)

    results.append(
        {
            "test_number": test_number,
            "metric": metric,
            "numerator_control": control_conversions,
            "denominator_control": control_sessions,
            "conversions_rate_control": control_cn,
            "numerator_test": experiment_conversions,
            "denominator_test": experiment_sessions,
            "conversions_rate_test": experiment_cn,
            "metric_change": metric_change,
            "z-stat": z_stat,
            "p-value": p_value,
            "significant": p_value < 0.05
        })

# Перетворюємо результат виконання циклу у таблицю pandas
result_df = pd.DataFrame(results)
print(result_df)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
         date  country   device continent         channel  test  test_group  \
0  2020-11-01  Ecuador  desktop  Americas  Organic Search     2           2   
1  2020-11-01  Ecuador  desktop  Americas  Organic Search     1           1   
2  2020-11-01  Uruguay   mobile  Americas  Organic Search     2           2   
3  2020-11-01  Uruguay   mobile  Americas  Organic Search     1           1   
4  2020-11-01    Kenya  desktop    Africa  Organic Search     2           1   
5  2020-11-01    Kenya  desktop    Africa  Organic Search     1           1   
6  2020-11-01   Serbia  desktop    Europe     Paid Search     2           1   
7  2020-11-01   Serbia  desktop    Europe     Paid Search     1           1   
8  2020-11-01    Malta   mobile    Europe   Social Search     2           2   
9  2020-11-01    Malta   mobile    Europe   Social Search     1           1   

 

Link to Tableau dashboard:  

https://public.tableau.com/views/ABtest_17790153012150/ABtests?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link

Link to csv file with results:
https://drive.google.com/file/d/1aKEKjaNaVsk7j4ilhr2vdbx-ZmulJE-b/view?usp=drive_link

In [ ]:
# Download file csv with results:
# result_df.to_csv(
#     'ab_test_significance_10.06.csv',
#     index=False
# )
# from google.colab import files

# files.download('ab_test_significance_10.06.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>